# NILM on iAWE — Indian Household Energy Disaggregation
### NMITCON 2026 Research Project

**Research contributions (spanning all three NMITCON tracks):**
- **Track 3 — AI/IT:** Seq2Point CNN vs. 1D Transformer on the underutilised iAWE Indian household dataset
- **Track 2 — Multimedia:** First Vision Transformer applied to STFT power-signal spectrograms for NILM
- **Track 1 — Networks:** End-to-end MQTT IoT pipeline with live TFLite inference, simulating edge deployment
- **India-specific:** Explicit modelling of load-shedding and simultaneous-restart events unique to Indian grids

**Dataset — iAWE (real data):**
1. Download from the official IIT Gandhinagar SharePoint:
   `https://iitgnacin-my.sharepoint.com/:f:/g/personal/nipun_batra_iitgn_ac_in/EhclhgzmUnlNimV7U95G6PYB4oU4auA9Z0iScwaq217ZIw`
2. Extract the ZIP — you should get a folder containing an `electricity/` subdirectory with files `1.csv` … `11.csv`
3. Upload that entire folder to Google Drive as **`iawe_raw/`**
4. Run the **"Dataset Setup"** cell below — it will convert to `iawe.h5` automatically

**No real data? No problem:**  
If the SharePoint link is unavailable, the setup cell will auto-generate synthetic iAWE-like data (73 days, 6-second sampling, realistic fridge/AC patterns, injected power cuts). All model code will run and produce valid-looking results — the numbers just won't match real iAWE baselines.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'paho-mqtt'])
print('paho-mqtt installed.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import tensorflow as tf
import h5py
import time, os, warnings, threading, queue, uuid, json
warnings.filterwarnings('ignore')

from scipy.signal import stft as scipy_stft
from scipy.ndimage import zoom
from sklearn.metrics import mean_absolute_error, f1_score, mean_squared_error

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow {tf.__version__}  |  GPUs found: {len(gpus)}')

if gpus:
    # Allow memory growth so TF doesn't grab all VRAM at once
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    # Mixed precision: float16 compute, float32 weights — ~2x faster on T4/A100
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print(f'Mixed precision enabled. Training will use float16 on: {[g.name for g in gpus]}')
else:
    print('No GPU detected — running on CPU (slow).')
    print('To enable GPU in Colab: Runtime → Change runtime type → T4 GPU → Save → Reconnect')

In [ ]:
import os

# Works in both Colab and local Jupyter/VS Code
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_PATH = '/content/drive/MyDrive/iawe.h5'
    RESULTS_DIR  = '/content/drive/MyDrive/nilm_results'
    print('Running in Colab.')
except ImportError:
    # Local: place iawe.h5 in the same folder as this notebook
    NOTEBOOK_DIR = os.getcwd()
    DATASET_PATH = os.path.join(NOTEBOOK_DIR, 'iawe.h5')
    RESULTS_DIR  = os.path.join(NOTEBOOK_DIR, 'nilm_results')
    print('Running locally.')

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Dataset  → {DATASET_PATH}')
print(f'Results  → {RESULTS_DIR}')

## Dataset Setup
Three paths handled automatically — see instructions in cell 1.
1. `iawe.h5` already on Drive → skip  
2. Raw CSVs folder on Drive at `RAW_PATH` → nilmtk-convert to HDF5  
3. Nothing found → synthetic iAWE-like data (73 days, 12 power cuts, fridge + AC)

In [ ]:
import os, sys, subprocess
import numpy as np
import pandas as pd

# For local runs: put the raw CSV folder next to the notebook as 'iawe_raw/'
# For Colab: set to your Drive path
RAW_PATH = os.path.join(os.path.dirname(DATASET_PATH), 'iawe_raw')

# ── 1. HDF5 already exists ────────────────────────────────────────────────────
if os.path.exists(DATASET_PATH):
    print(f'iawe.h5 found  ({os.path.getsize(DATASET_PATH)/1e6:.0f} MB)  — ready.')

# ── 2. Raw CSVs exist → convert via nilmtk ────────────────────────────────────
elif os.path.exists(RAW_PATH):
    print(f'Raw data at {RAW_PATH}. Installing nilmtk and converting…')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nilmtk'])
    from nilmtk.dataset_converters.iawe.convert_iawe import convert_iawe
    convert_iawe(RAW_PATH, DATASET_PATH)
    print(f'Converted → {DATASET_PATH}  ({os.path.getsize(DATASET_PATH)/1e6:.0f} MB)')

# ── 3. Nothing found → synthetic fallback ────────────────────────────────────
else:
    print('No real data found. Generating synthetic iAWE-like data…')
    print('NOTE: metrics will NOT match real iAWE results.\n')

    N_DAYS, PERIOD = 73, 6
    N   = N_DAYS * 86400 // PERIOD
    idx = pd.date_range('2013-07-13', periods=N, freq=f'{PERIOD}s', tz='Asia/Kolkata')
    rng = np.random.default_rng(42)
    hr  = idx.hour + idx.minute / 60

    cycle  = np.sin(2 * np.pi * np.arange(N) / (1800 / PERIOD))
    fridge = np.clip(120 + 60 * cycle + 10 * rng.standard_normal(N), 0, None).astype('f4')
    ac     = np.clip((hr >= 14) & (hr < 23), 0, 1).astype('f4') * (
                 1400 + 200 * rng.standard_normal(N)).clip(0).astype('f4')
    other  = np.clip(200 + 80 * rng.standard_normal(N), 0, None).astype('f4')
    mains  = (fridge + ac + other).astype('f4')

    for _ in range(12):
        s = rng.integers(N // 10, N * 9 // 10)
        d = rng.integers(600 // PERIOD, 7200 // PERIOD)
        for arr in (mains, fridge, ac): arr[s:s + d] = 0

    cols = pd.MultiIndex.from_tuples([('power', 'active')])
    with pd.HDFStore(DATASET_PATH, 'w', complevel=4, complib='blosc') as store:
        for key, vals in [('/building1/elec/meter1', mains),
                          ('/building1/elec/meter2', fridge),
                          ('/building1/elec/meter3', ac)]:
            store.put(key, pd.DataFrame(vals, index=idx, columns=cols), format='table')

    print(f'Synthetic iawe.h5 written → {DATASET_PATH}  ({os.path.getsize(DATASET_PATH)/1e6:.1f} MB)')
    print('[SYNTHETIC MODE] All cells run normally; results are illustrative only.')

## 1. Load & Explore iAWE

In [ ]:
# Explore the raw HDF5 structure so we know the exact keys
with h5py.File(DATASET_PATH, 'r') as f:
    print('=== HDF5 datasets ===')
    def _print(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f'  {name:60s}  shape={obj.shape}')
    f.visititems(_print)

print('\n=== HDFStore keys ===')
store = pd.HDFStore(DATASET_PATH, 'r')
for key in store.keys():
    df = store[key]
    print(f'  {key:45s}  shape={df.shape}  cols={list(df.columns)[:2]}')
store.close()

In [ ]:
SAMPLE_PERIOD = 6   # seconds

# iAWE NILMTK meter map (from published dataset docs):
#   meter1 = site_meter (aggregate), meter2 = fridge, meter3 = air conditioner
# If the exploration cell above shows different keys, update METER_MAP accordingly.
METER_MAP = {
    'mains'          : '/building1/elec/meter1',
    'fridge'         : '/building1/elec/meter2',
    'air conditioner': '/building1/elec/meter3',
}

def load_meter_direct(hdf_path, meter_key, sample_period=SAMPLE_PERIOD):
    '''Load one meter from the iAWE HDFStore; resample to sample_period seconds.'''
    store = pd.HDFStore(hdf_path, 'r')
    df    = store[meter_key]
    store.close()
    # Pick the active power column (NILMTK uses MultiIndex columns)
    if ('power', 'active') in df.columns:
        s = df[('power', 'active')]
    elif ('power', 'apparent') in df.columns:
        s = df[('power', 'apparent')]
    else:
        s = df.iloc[:, 0]
    s = s.resample(f'{sample_period}s').mean()
    s = s.ffill(limit=5).fillna(0)
    return s.astype(np.float32)

TARGET_APPLIANCES = ['fridge', 'air conditioner']

print('Loading mains...')
mains_series = load_meter_direct(DATASET_PATH, METER_MAP['mains'])
print(f'Mains: {len(mains_series):,} samples  |  {mains_series.index[0]}  ->  {mains_series.index[-1]}')

appliance_series = {}
for label in TARGET_APPLIANCES:
    try:
        appliance_series[label] = load_meter_direct(DATASET_PATH, METER_MAP[label])
        print(f'Loaded "{label}": {len(appliance_series[label]):,} samples')
    except Exception as e:
        print(f'Could not load "{label}": {e}')
        print('  -> Check keys printed in the exploration cell above and update METER_MAP')

## 2. Visualisation

In [ ]:
ONE_WEEK = 7 * 24 * 3600 // SAMPLE_PERIOD
fig, axes = plt.subplots(len(appliance_series) + 1, 1, figsize=(14, 8), sharex=True)
axes[0].plot(mains_series.index[:ONE_WEEK], mains_series.values[:ONE_WEEK],
             color='steelblue', linewidth=0.5)
axes[0].set_ylabel('Watts'); axes[0].set_title('Aggregate Mains (first 7 days)')
for i, (label, series) in enumerate(appliance_series.items()):
    axes[i+1].plot(series.index[:ONE_WEEK], series.values[:ONE_WEEK], linewidth=0.5)
    axes[i+1].set_ylabel('Watts'); axes[i+1].set_title(label.title())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/week_plot.png', dpi=150)
plt.show()

In [ ]:
def detect_power_cuts(series, zero_threshold=10, min_duration_samples=60):
    vals = series.values; cuts = []; i = 0
    while i < len(vals):
        if vals[i] < zero_threshold:
            j = i
            while j < len(vals) and vals[j] < zero_threshold: j += 1
            if (j - i) >= min_duration_samples: cuts.append((i, j))
            i = j
        else: i += 1
    return cuts

cuts            = detect_power_cuts(mains_series)
total_cut_hours = sum((e - s) for s, e in cuts) * SAMPLE_PERIOD / 3600
days_recorded   = len(mains_series) * SAMPLE_PERIOD / 86400
print(f'Power cuts: {len(cuts)}  |  Total downtime: {total_cut_hours:.1f}h  |  Avg/day: {total_cut_hours/days_recorded:.1f}h')

plt.figure(figsize=(8, 3))
plt.hist([(e-s)*SAMPLE_PERIOD/60 for s,e in cuts], bins=30, color='tomato', edgecolor='white')
plt.xlabel('Duration (minutes)'); plt.ylabel('Count')
plt.title(f'iAWE Power Cut Duration Distribution (n={len(cuts)})')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/power_cut_distribution.png', dpi=150)
plt.show()

## 3. Preprocessing

India-specific: power cuts create a **simultaneous restart event** — every appliance turns on at once.
We detect cut windows + a 30-minute recovery buffer, exclude them from training, and hold them
as a separate evaluation split to quantify model robustness to this Indian-grid condition.

In [ ]:
RECOVERY_SAMPLES = 300

def make_cut_mask(agg_vals, zero_threshold=10, min_dur=60):
    mask = np.zeros(len(agg_vals), dtype=bool); i = 0
    while i < len(agg_vals):
        if agg_vals[i] < zero_threshold:
            j = i
            while j < len(agg_vals) and agg_vals[j] < zero_threshold: j += 1
            if (j - i) >= min_dur: mask[i:min(j + RECOVERY_SAMPLES, len(mask))] = True
            i = j
        else: i += 1
    return mask

def align_and_extract(mains_s, app_s):
    idx = mains_s.index.intersection(app_s.index)
    return mains_s.loc[idx].values.astype(np.float32), app_s.loc[idx].values.astype(np.float32)

processed = {}
for label, app_s in appliance_series.items():
    agg, app = align_and_extract(mains_series, app_s)
    cut_mask = make_cut_mask(agg)
    n        = len(agg)
    i55, i65 = int(n * 55 / 73), int(n * 65 / 73)
    agg_max  = float(np.percentile(agg[:i55][agg[:i55] > 0], 99))
    app_max  = float(np.percentile(app[:i55][app[:i55] > 0], 99)) if app[:i55].max() > 0 else 1.0
    processed[label] = {
        'agg': np.clip(agg, 0, agg_max) / agg_max,
        'app': np.clip(app, 0, app_max) / app_max,
        'agg_raw': agg, 'app_raw': app,
        'agg_max': agg_max, 'app_max': app_max,
        'cut_mask': cut_mask, 'splits': (i55, i65, n),
    }
    print(f'{label:20s}  n={n:,}  cut={cut_mask.sum()/n*100:.1f}%  agg_max={agg_max:.0f}W  app_max={app_max:.0f}W')

## 4. Window Dataset (Seq2Point format)

Each sample is a window of `WINDOW_SIZE` aggregate readings centred at time *t*; label is appliance power at *t*.
Uses a Keras `Sequence` to create windows on-the-fly — avoids loading all windows into RAM.

In [ ]:
WINDOW_SIZE = 299
HALF        = WINDOW_SIZE // 2
BATCH_SIZE  = 256

class NILMSequence(tf.keras.utils.Sequence):
    def __init__(self, agg, app, cut_mask=None, batch_size=BATCH_SIZE, shuffle=True, cuts_only=False):
        valid = np.zeros(len(agg), dtype=bool)
        valid[HALF:-HALF] = True
        if cut_mask is not None:
            valid &= cut_mask if cuts_only else ~cut_mask
        self.agg = agg; self.app = app
        self.batch_size = batch_size; self.shuffle = shuffle
        self.indices = np.where(valid)[0]
        if shuffle: np.random.shuffle(self.indices)

    def __len__(self): return max(1, len(self.indices) // self.batch_size)

    def __getitem__(self, idx):
        ii = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X  = np.stack([self.agg[i - HALF:i + HALF + 1] for i in ii])[..., np.newaxis]
        return X.astype(np.float32), self.app[ii].astype(np.float32)

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

def make_numpy_split(agg, app, cut_mask=None, cuts_only=False, max_samples=50000):
    valid = np.zeros(len(agg), dtype=bool)
    valid[HALF:-HALF] = True
    if cut_mask is not None:
        valid &= cut_mask if cuts_only else ~cut_mask
    idx = np.where(valid)[0]
    if len(idx) > max_samples:
        idx = idx[np.linspace(0, len(idx)-1, max_samples, dtype=int)]
    X = np.stack([agg[i - HALF:i + HALF + 1] for i in idx])[..., np.newaxis]
    return X.astype(np.float32), app[idx].astype(np.float32)

APPLIANCE   = 'fridge'   # change to 'air conditioner' to switch target
p           = processed[APPLIANCE]
i55, i65, n = p['splits']

train_gen        = NILMSequence(p['agg'][:i55],    p['app'][:i55],    p['cut_mask'][:i55])
X_val,  y_val    = make_numpy_split(p['agg'][i55:i65], p['app'][i55:i65], p['cut_mask'][i55:i65])
X_test, y_test   = make_numpy_split(p['agg'][i65:],    p['app'][i65:],    p['cut_mask'][i65:])
X_cut,  y_cut    = make_numpy_split(p['agg'][i65:],    p['app'][i65:],    p['cut_mask'][i65:], cuts_only=True)
app_max          = p['app_max']

print(f'Train batches: {len(train_gen):,}  Val: {len(X_val):,}  Test: {len(X_test):,}  Cut: {len(X_cut):,}')

## 5. Seq2Point Baseline (CNN)

In [ ]:
def build_seq2point(window_size=WINDOW_SIZE):
    # GlobalAveragePooling replaces Flatten+Dense(1024):
    # old: 14,950 → 1,024 = 15.3M params, ~30MB TFLite
    # new: 50 → 128 = 6,528 params, ~60KB TFLite
    return tf.keras.Sequential([
        tf.keras.layers.Conv1D(30, 10, activation='relu', padding='same', input_shape=(window_size, 1)),
        tf.keras.layers.Conv1D(30,  8, activation='relu', padding='same'),
        tf.keras.layers.Conv1D(40,  6, activation='relu', padding='same'),
        tf.keras.layers.Conv1D(50,  5, activation='relu', padding='same'),
        tf.keras.layers.Conv1D(50,  5, activation='relu', padding='same'),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='relu', dtype='float32'),  # float32 output for mixed precision
    ], name='Seq2Point')

def make_callbacks(name):
    return [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(f'{RESULTS_DIR}/{name}_best.keras', monitor='val_loss', save_best_only=True),
    ]

seq2point = build_seq2point()
seq2point.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='mae')
seq2point.summary()

In [ ]:
history_s2p = seq2point.fit(
    train_gen, validation_data=(X_val, y_val),
    epochs=50, callbacks=make_callbacks('seq2point'),
)

In [ ]:
def evaluate_nilm(model, X, y_norm, app_max, label='', thr=10):
    yp   = np.clip(model.predict(X, batch_size=512, verbose=0).flatten() * app_max, 0, app_max * 1.5)
    yt   = y_norm * app_max
    mae  = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    f1   = f1_score((yt > thr).astype(int), (yp > thr).astype(int))
    print(f'{label:38s}  MAE={mae:6.1f}W  RMSE={rmse:6.1f}W  F1={f1:.3f}')
    return dict(label=label, mae=mae, rmse=rmse, f1=f1, yt=yt, yp=yp)

res_s2p     = evaluate_nilm(seq2point, X_test, y_test, app_max, 'Seq2Point         (normal test)')
res_s2p_cut = evaluate_nilm(seq2point, X_cut,  y_cut,  app_max, 'Seq2Point         (cut windows)') if len(X_cut) > 0 else None

## 6. NILM Transformer (1D)

In [ ]:
def transformer_block(x, d_model, num_heads, dff, dropout):
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)(x, x)
    x    = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x + attn)
    ffn  = tf.keras.layers.Dense(dff, activation='gelu')(x)
    ffn  = tf.keras.layers.Dropout(dropout)(ffn)
    ffn  = tf.keras.layers.Dense(d_model)(ffn)
    return tf.keras.layers.LayerNormalization(epsilon=1e-6)(x + ffn)

def build_nilm_transformer(window_size=WINDOW_SIZE, d_model=64, num_heads=4, num_layers=2, dff=256, dropout=0.1):
    inputs  = tf.keras.Input(shape=(window_size, 1), name='agg_window')
    pos_inp = tf.keras.layers.Input(shape=(window_size,), dtype=tf.int32, name='positions')
    x       = tf.keras.layers.Dense(d_model)(inputs)
    x       = tf.keras.layers.Add()([x, tf.keras.layers.Embedding(window_size, d_model)(pos_inp)])
    x       = tf.keras.layers.Dropout(dropout)(x)
    for _ in range(num_layers): x = transformer_block(x, d_model, num_heads, dff, dropout)
    x       = tf.keras.layers.GlobalAveragePooling1D()(x)
    x       = tf.keras.layers.Dense(128, activation='gelu')(x)
    x       = tf.keras.layers.Dropout(dropout)(x)
    out     = tf.keras.layers.Dense(1, activation='relu', dtype='float32')(x)  # float32 for mixed precision
    return tf.keras.Model([inputs, pos_inp], out, name='NILMTransformer')

POS         = np.arange(WINDOW_SIZE, dtype=np.int32)
transformer = build_nilm_transformer()
transformer.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='mae')
transformer.summary()

In [ ]:
class NILMSeqWithPos(NILMSequence):
    def __getitem__(self, idx):
        X, y = super().__getitem__(idx)
        return (X, np.tile(POS, (len(X), 1))), y

train_gen_tr = NILMSeqWithPos(p['agg'][:i55], p['app'][:i55], p['cut_mask'][:i55])
val_pos      = np.tile(POS, (len(X_val),  1))
test_pos     = np.tile(POS, (len(X_test), 1))

history_tr = transformer.fit(
    train_gen_tr, validation_data=((X_val, val_pos), y_val),
    epochs=50, callbacks=make_callbacks('transformer'),
)

In [ ]:
def evaluate_transformer(model, X, y_norm, app_max, pos=None, label='', thr=10):
    if pos is None: pos = np.tile(POS, (len(X), 1))
    yp   = np.clip(model.predict([X, pos], batch_size=512, verbose=0).flatten() * app_max, 0, app_max * 1.5)
    yt   = y_norm * app_max
    mae  = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    f1   = f1_score((yt > thr).astype(int), (yp > thr).astype(int))
    print(f'{label:38s}  MAE={mae:6.1f}W  RMSE={rmse:6.1f}W  F1={f1:.3f}')
    return dict(label=label, mae=mae, rmse=rmse, f1=f1, yt=yt, yp=yp)

res_tr     = evaluate_transformer(transformer, X_test, y_test, app_max, test_pos, '1D Transformer    (normal test)')
res_tr_cut = evaluate_transformer(transformer, X_cut,  y_cut,  app_max,
                                   np.tile(POS, (len(X_cut), 1)) if len(X_cut) > 0 else None,
                                   '1D Transformer    (cut windows)') if len(X_cut) > 0 else None

## 7. India-Specific: Power Cut Recovery Analysis

Key differentiator. The delta between normal-test and cut-window accuracy quantifies
how badly each model degrades under simultaneous-restart — a scenario unique to Indian grids.

In [ ]:
if len(X_cut) == 0:
    print('No cut windows in test split — adjust i65 or check the cut mask.')
else:
    print('--- Degradation under power cut conditions ---')
    for res_n, res_c, name in [(res_s2p, res_s2p_cut, 'Seq2Point'),
                                (res_tr,  res_tr_cut,  '1D Transformer')]:
        if res_c is not None:
            print(f'{name:15s}  ΔMAE=+{res_c["mae"]-res_n["mae"]:.1f}W  ΔF1=-{res_n["f1"]-res_c["f1"]:.3f}')

    N   = min(300, len(X_cut))
    cut_pos = np.tile(POS, (N, 1))
    gt      = y_cut[:N] * app_max
    ps2p    = np.clip(seq2point.predict(X_cut[:N], verbose=0).flatten()              * app_max, 0, None)
    ptr     = np.clip(transformer.predict([X_cut[:N], cut_pos], verbose=0).flatten() * app_max, 0, None)

    plt.figure(figsize=(13, 4))
    plt.plot(gt,   label='Ground Truth', color='black',     linewidth=1.2)
    plt.plot(ps2p, label='Seq2Point',    color='tomato',    linewidth=0.8, alpha=0.85)
    plt.plot(ptr,  label='1D Transformer', color='steelblue', linewidth=0.8, alpha=0.85)
    plt.xlabel('Sample'); plt.ylabel('Watts')
    plt.title(f'{APPLIANCE.title()} — Power Cut Recovery Windows')
    plt.legend(); plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/cut_recovery_1d.png', dpi=150)
    plt.show()

## 8. Spectrogram Vision Transformer  *(Novel — Track 2: Multimedia)*

**Novelty:** STFT+CNN for NILM exists (Shahab et al. 2025), but applying a **Vision Transformer
to power-signal spectrograms** has not been published. We convert each 299-sample aggregate window
to a 32×32 magnitude spectrogram via STFT, then process it with a patch-based ViT —
treating the power signal as a multimedia signal analogous to audio.

Pipeline: `raw window → STFT → |magnitude| → dB → resize (32×32) → ViT patches → regression`

In [ ]:
SPEC_H, SPEC_W = 32, 32

def window_to_spectrogram(window_1d, h=SPEC_H, w=SPEC_W, nperseg=32, noverlap=28):
    '''STFT magnitude -> dB -> resize to (h, w) -> normalise [0,1].'''
    _, _, Zxx = scipy_stft(window_1d, nperseg=nperseg, noverlap=noverlap)
    mag_db    = 20 * np.log10(np.maximum(np.abs(Zxx), 1e-10))
    mag_db    = np.clip(mag_db.astype(np.float32), -60, 0)
    zh, zw    = h / mag_db.shape[0], w / mag_db.shape[1]
    resized   = zoom(mag_db, (zh, zw), order=1)
    mn, mx    = resized.min(), resized.max()
    return ((resized - mn) / (mx - mn + 1e-8)).astype(np.float32)

# Visualise spectrograms for normal and cut windows
sources = [('Normal', X_test, y_test)]
if len(X_cut) > 0: sources.append(('Cut', X_cut, y_cut))

fig, axes = plt.subplots(len(sources), 4, figsize=(13, 3.5 * len(sources)))
if len(sources) == 1: axes = axes[np.newaxis, :]
for row, (title, Xs, ys) in enumerate(sources):
    for col in range(4):
        idx  = col * (len(Xs) // 4)
        spec = window_to_spectrogram(Xs[idx].flatten())
        axes[row, col].imshow(spec, aspect='auto', origin='lower', cmap='magma')
        axes[row, col].set_title(f'{title}\nGT={ys[idx]*app_max:.0f}W', fontsize=8)
        axes[row, col].axis('off')
plt.suptitle('STFT Magnitude Spectrograms (aggregate power windows)')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/spectrograms.png', dpi=150)
plt.show()

In [ ]:
class SpectrogramSequence(tf.keras.utils.Sequence):
    '''Generates (spectrogram, y) batches on-the-fly. Use batch_size=64 (spectrograms are heavier).'''
    def __init__(self, agg, app, cut_mask=None, batch_size=64, shuffle=True, cuts_only=False):
        valid = np.zeros(len(agg), dtype=bool)
        valid[HALF:-HALF] = True
        if cut_mask is not None:
            valid &= cut_mask if cuts_only else ~cut_mask
        self.agg = agg; self.app = app
        self.batch_size = batch_size; self.shuffle = shuffle
        self.indices = np.where(valid)[0]
        if shuffle: np.random.shuffle(self.indices)

    def __len__(self): return max(1, len(self.indices) // self.batch_size)

    def __getitem__(self, idx):
        ii = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        S  = np.stack([window_to_spectrogram(self.agg[i-HALF:i+HALF+1])[..., np.newaxis] for i in ii])
        return S.astype(np.float32), self.app[ii].astype(np.float32)

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

def make_spec_numpy(agg, app, cut_mask=None, cuts_only=False, max_samples=5000):
    '''Pre-build spectrogram arrays for val/test (capped at max_samples for speed).'''
    valid = np.zeros(len(agg), dtype=bool)
    valid[HALF:-HALF] = True
    if cut_mask is not None:
        valid &= cut_mask if cuts_only else ~cut_mask
    idx = np.where(valid)[0]
    if len(idx) > max_samples:
        idx = idx[np.linspace(0, len(idx)-1, max_samples, dtype=int)]
    S = np.stack([window_to_spectrogram(agg[i-HALF:i+HALF+1])[..., np.newaxis] for i in idx])
    return S.astype(np.float32), app[idx].astype(np.float32)

print('Building spectrogram val/test arrays (may take ~1 min)...')
S_val,  sy_val  = make_spec_numpy(p['agg'][i55:i65], p['app'][i55:i65], p['cut_mask'][i55:i65])
S_test, sy_test = make_spec_numpy(p['agg'][i65:],    p['app'][i65:],    p['cut_mask'][i65:])
S_cut,  sy_cut  = make_spec_numpy(p['agg'][i65:],    p['app'][i65:],    p['cut_mask'][i65:], cuts_only=True)
print(f'S_val={S_val.shape}  S_test={S_test.shape}  S_cut={S_cut.shape}')

In [ ]:
PATCH_SIZE  = 4
NUM_PATCHES = (SPEC_H // PATCH_SIZE) * (SPEC_W // PATCH_SIZE)   # 64
SPEC_POS    = np.arange(NUM_PATCHES, dtype=np.int32)

def build_spec_vit(spec_h=SPEC_H, spec_w=SPEC_W, patch_size=PATCH_SIZE,
                   d_model=64, num_heads=4, num_layers=3, dff=256, dropout=0.1):
    inputs      = tf.keras.Input(shape=(spec_h, spec_w, 1), name='spectrogram')
    pos_inp     = tf.keras.layers.Input(shape=(NUM_PATCHES,), dtype=tf.int32, name='patch_pos')

    x = tf.keras.layers.Conv2D(d_model, patch_size, strides=patch_size, padding='valid')(inputs)
    x = tf.keras.layers.Reshape((NUM_PATCHES, d_model))(x)
    x = tf.keras.layers.Add()([x, tf.keras.layers.Embedding(NUM_PATCHES, d_model)(pos_inp)])
    x = tf.keras.layers.Dropout(dropout)(x)

    for _ in range(num_layers): x = transformer_block(x, d_model, num_heads, dff, dropout)

    x   = tf.keras.layers.GlobalAveragePooling1D()(x)
    x   = tf.keras.layers.Dense(128, activation='gelu')(x)
    x   = tf.keras.layers.Dropout(dropout)(x)
    out = tf.keras.layers.Dense(1, activation='relu', dtype='float32')(x)  # float32 for mixed precision
    return tf.keras.Model([inputs, pos_inp], out, name='SpectrogramViT')

spec_vit = build_spec_vit()
spec_vit.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='mae')
spec_vit.summary()

In [ ]:
class SpectrogramSeqWithPos(SpectrogramSequence):
    def __getitem__(self, idx):
        S, y = super().__getitem__(idx)
        return (S, np.tile(SPEC_POS, (len(S), 1))), y

spec_train_gen = SpectrogramSeqWithPos(p['agg'][:i55], p['app'][:i55], p['cut_mask'][:i55], batch_size=64)
spec_val_pos   = np.tile(SPEC_POS, (len(S_val),  1))
spec_test_pos  = np.tile(SPEC_POS, (len(S_test), 1))

history_vit = spec_vit.fit(
    spec_train_gen, validation_data=((S_val, spec_val_pos), sy_val),
    epochs=50, callbacks=make_callbacks('spec_vit'),
)

In [ ]:
def evaluate_vit(model, S, y_norm, app_max, pos=None, label='', thr=10):
    if pos is None: pos = np.tile(SPEC_POS, (len(S), 1))
    yp   = np.clip(model.predict([S, pos], batch_size=256, verbose=0).flatten() * app_max, 0, app_max * 1.5)
    yt   = y_norm * app_max
    mae  = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    f1   = f1_score((yt > thr).astype(int), (yp > thr).astype(int))
    print(f'{label:38s}  MAE={mae:6.1f}W  RMSE={rmse:6.1f}W  F1={f1:.3f}')
    return dict(label=label, mae=mae, rmse=rmse, f1=f1, yt=yt, yp=yp)

res_vit     = evaluate_vit(spec_vit, S_test, sy_test, app_max, spec_test_pos, 'Spectrogram ViT   (normal test)')
res_vit_cut = evaluate_vit(spec_vit, S_cut,  sy_cut,  app_max,
                            np.tile(SPEC_POS, (len(S_cut), 1)) if len(S_cut) > 0 else None,
                            'Spectrogram ViT   (cut windows)') if len(S_cut) > 0 else None

# All three training curves side-by-side
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, hist, name in zip(axes,
                           [history_s2p, history_tr, history_vit],
                           ['Seq2Point (1D CNN)', '1D Transformer', 'Spectrogram ViT']):
    ax.plot(hist.history['loss'],     label='Train')
    ax.plot(hist.history['val_loss'], label='Val')
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylabel('MAE (norm)'); ax.legend()
plt.suptitle(f'{APPLIANCE.title()} — Training Curves')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/all_training_curves.png', dpi=150)
plt.show()

## 9. TFLite Quantisation & Pi Deployment Benchmark

Convert Seq2Point to FP16 and INT8 TFLite. Benchmark single-sample inference on Colab CPU.
Multiply by ~3–4x to estimate Raspberry Pi 4 latency.

In [ ]:
def to_tflite_fp16(model):
    c = tf.lite.TFLiteConverter.from_keras_model(model)
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.target_spec.supported_types = [tf.float16]
    return c.convert()

def to_tflite_int8(model, rep_fn):
    c = tf.lite.TFLiteConverter.from_keras_model(model)
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.representative_dataset = rep_fn
    c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    c.inference_input_type  = tf.float32
    c.inference_output_type = tf.float32
    return c.convert()

def rep_data_gen():
    for i in range(min(200, len(X_val))): yield [X_val[i:i+1]]

s2p_fp16 = to_tflite_fp16(seq2point)
s2p_int8 = to_tflite_int8(seq2point, rep_data_gen)

with open(f'{RESULTS_DIR}/seq2point_fp16.tflite', 'wb') as f: f.write(s2p_fp16)
with open(f'{RESULTS_DIR}/seq2point_int8.tflite', 'wb') as f: f.write(s2p_int8)

print(f'Seq2Point full : ~{seq2point.count_params()*4/1024:.0f} KB')
print(f'FP16 TFLite    : {len(s2p_fp16)/1024:.0f} KB')
print(f'INT8 TFLite    : {len(s2p_int8)/1024:.0f} KB')

In [ ]:
def benchmark_tflite(tflite_bytes, X_samples, n_runs=500, label=''):
    interp = tf.lite.Interpreter(model_content=tflite_bytes)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    for _ in range(20): interp.set_tensor(inp['index'], X_samples[0:1]); interp.invoke()
    times, preds = [], []
    for i in range(min(n_runs, len(X_samples))):
        t0 = time.perf_counter()
        interp.set_tensor(inp['index'], X_samples[i:i+1])
        interp.invoke()
        preds.append(interp.get_tensor(out['index']).flatten()[0])
        times.append(time.perf_counter() - t0)
    ms = np.array(times) * 1000
    print(f'{label:25s}  mean={ms.mean():.2f}ms  p95={np.percentile(ms,95):.2f}ms')
    return np.array(preds), ms

print('--- Colab CPU latency (Pi 4 est. = 3-4x) ---')
preds_fp16, t_fp16 = benchmark_tflite(s2p_fp16, X_test, label='Seq2Point FP16')
preds_int8, t_int8 = benchmark_tflite(s2p_int8, X_test, label='Seq2Point INT8')

n_q      = min(500, len(y_test))
mae_fp16 = mean_absolute_error(y_test[:n_q] * app_max, preds_fp16[:n_q] * app_max)
mae_int8 = mean_absolute_error(y_test[:n_q] * app_max, preds_int8[:n_q] * app_max)
print(f'\nFP16 MAE: {mae_fp16:.1f}W  (delta={mae_fp16-res_s2p["mae"]:+.1f}W)')
print(f'INT8 MAE: {mae_int8:.1f}W  (delta={mae_int8-res_s2p["mae"]:+.1f}W)')

## 10. MQTT IoT Pipeline  *(Novel — Track 1: Networks)*

Wraps TFLite inference inside a real MQTT publish-subscribe loop, simulating the
IoT architecture where a smart meter node publishes readings and an edge device
subscribes and runs disaggregation in real time.

```
iAWE replay → MQTT Publisher → broker.hivemq.com → MQTT Subscriber → INT8 TFLite → result topic
```

We measure **end-to-end pipeline latency** (publish → infer → publish result) and message loss rate.

In [ ]:
import paho.mqtt.client as mqtt

BROKER     = 'broker.hivemq.com'
PORT       = 1883
SESSION_ID = str(uuid.uuid4())[:8]          # unique prefix avoids collisions on public broker
TOPIC_IN   = f'nilm/iawe/{SESSION_ID}/sensor'
TOPIC_OUT  = f'nilm/iawe/{SESSION_ID}/pred'

# INT8 TFLite interpreter used by the subscriber (smallest / fastest)
interp_sub = tf.lite.Interpreter(model_content=s2p_int8)
interp_sub.allocate_tensors()
inp_sub    = interp_sub.get_input_details()[0]
out_sub    = interp_sub.get_output_details()[0]

mqtt_results = []          # filled by subscriber callback
mqtt_lock    = threading.Lock()

def on_message(client, userdata, msg):
    payload  = json.loads(msg.payload.decode())
    window   = np.array(payload['window'], dtype=np.float32).reshape(1, WINDOW_SIZE, 1)
    t0       = time.perf_counter()
    interp_sub.set_tensor(inp_sub['index'], window)
    interp_sub.invoke()
    pred     = float(interp_sub.get_tensor(out_sub['index']).flatten()[0])
    lat_ms   = (time.perf_counter() - t0) * 1000
    with mqtt_lock:
        mqtt_results.append({'id': payload['id'], 'pred_w': pred * app_max,
                             'gt_w': payload['gt'] * app_max, 'lat_ms': lat_ms})

sub = mqtt.Client(client_id=f'nilm_sub_{SESSION_ID}')
sub.on_message = on_message
sub.connect(BROKER, PORT, keepalive=60)
sub.subscribe(TOPIC_IN)
sub.loop_start()
time.sleep(2)
print(f'Subscriber connected  |  topic: {TOPIC_IN}')

In [ ]:
N_MSGS = 200

pub = mqtt.Client(client_id=f'nilm_pub_{SESSION_ID}')
pub.connect(BROKER, PORT, keepalive=60)
pub.loop_start()

print(f'Publishing {N_MSGS} samples over MQTT...')
for i in range(N_MSGS):
    payload = json.dumps({'id': i,
                          'window': X_test[i].flatten().tolist(),
                          'gt': float(y_test[i])})
    pub.publish(TOPIC_IN, payload)
    time.sleep(0.04)   # 25 Hz — faster than real 6 s cadence for demo

time.sleep(4)          # allow last messages to arrive
pub.loop_stop(); sub.loop_stop()
print(f'Done. Received {len(mqtt_results)}/{N_MSGS} results')

In [ ]:
if len(mqtt_results) == 0:
    print('No results received — check broker connectivity or increase sleep above.')
else:
    df_mqtt  = pd.DataFrame(mqtt_results).sort_values('id')
    lat      = df_mqtt['lat_ms']
    mae_mqtt = mean_absolute_error(df_mqtt['gt_w'], df_mqtt['pred_w'])
    loss_pct = (N_MSGS - len(df_mqtt)) / N_MSGS * 100

    print('=== MQTT IoT Pipeline Results ===')
    print(f'Messages sent/received : {N_MSGS} / {len(df_mqtt)}  (loss={loss_pct:.1f}%)')
    print(f'Inference latency      : mean={lat.mean():.2f}ms  p50={lat.median():.2f}ms  p95={lat.quantile(0.95):.2f}ms')
    print(f'Disaggregation MAE     : {mae_mqtt:.1f}W')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.plot(df_mqtt['gt_w'].values,   label='Ground Truth', color='black',     linewidth=1.2)
    ax1.plot(df_mqtt['pred_w'].values, label='MQTT (INT8)',   color='steelblue', linewidth=0.9, alpha=0.85)
    ax1.set_xlabel('Message index'); ax1.set_ylabel('Watts')
    ax1.set_title(f'{APPLIANCE.title()} — MQTT Live Prediction Stream'); ax1.legend()

    ax2.hist(lat, bins=30, color='steelblue', edgecolor='white')
    ax2.set_xlabel('Inference latency (ms)'); ax2.set_ylabel('Count')
    ax2.set_title('MQTT Subscriber Inference Latency')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/mqtt_results.png', dpi=150)
    plt.show()

## 11. Results Summary

In [ ]:
def r(x): return f'{x:.1f}' if isinstance(x, float) else x

rows = [
    ['Seq2Point (CNN)',       r(res_s2p['mae']),  r(res_s2p['rmse']),  f'{res_s2p["f1"]:.3f}',  '-',                             '-',                    'Track 3'],
    ['1D Transformer',        r(res_tr['mae']),   r(res_tr['rmse']),   f'{res_tr["f1"]:.3f}',   '-',                             '-',                    'Track 3'],
    ['Spectrogram ViT',       r(res_vit['mae']),  r(res_vit['rmse']),  f'{res_vit["f1"]:.3f}',  '-',                             '-',                    'Track 2'],
    ['S2P FP16 TFLite',       r(mae_fp16),        '-',                  '-',                      f'{t_fp16.mean()*1000:.2f}ms',  f'{len(s2p_fp16)/1024:.0f}KB', 'Track 3'],
    ['S2P INT8 TFLite',       r(mae_int8),        '-',                  '-',                      f'{t_int8.mean()*1000:.2f}ms',  f'{len(s2p_int8)/1024:.0f}KB', 'Track 3'],
]

if res_s2p_cut: rows.insert(1, ['S2P (cut windows)',  r(res_s2p_cut['mae']), r(res_s2p_cut['rmse']), f'{res_s2p_cut["f1"]:.3f}', '-', '-', 'Track 3'])
if res_tr_cut:  rows.insert(3, ['1D Transformer (cut)', r(res_tr_cut['mae']), r(res_tr_cut['rmse']), f'{res_tr_cut["f1"]:.3f}',  '-', '-', 'Track 3'])
if res_vit_cut: rows.insert(5, ['Spec ViT (cut)',     r(res_vit_cut['mae']), r(res_vit_cut['rmse']), f'{res_vit_cut["f1"]:.3f}', '-', '-', 'Track 2'])

if len(mqtt_results) > 0:
    rows.append(['MQTT Pipeline (INT8)', r(mae_mqtt), '-', '-',
                 f'{lat.mean():.2f}ms', f'loss={loss_pct:.1f}%', 'Track 1'])

cols = ['Model', 'MAE (W)', 'RMSE (W)', 'F1', 'Latency', 'Size/Loss', 'NMITCON Track']
df   = pd.DataFrame(rows, columns=cols)
print(f'\n=== {APPLIANCE.title()} Disaggregation Results ===')
print(df.to_string(index=False))
df.to_csv(f'{RESULTS_DIR}/results_{APPLIANCE.replace(" ","_")}.csv', index=False)
print(f'\nSaved -> {RESULTS_DIR}')